# Does transaction amount predict fraud?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
import polars as pl
from IPython.display import Markdown, display

t = pl.scan_csv("../../kaggle/raw/train_transaction.csv")
ctx = pl.SQLContext()
ctx.register("t", t)


<SQLContext [tables:1] at 0x1119e5400>

In [2]:
import polars as pl

# Use Polars lazy scanning
t = pl.scan_csv("../../kaggle/raw/train_transaction.csv")
i = pl.scan_csv("../../kaggle/raw/train_identity.csv")

ctx = pl.SQLContext()
ctx.register("t", t)
ctx.register("i", i)

query = """
SELECT 
    COUNT(*) as Transactions,
    SUM(isFraud) as Fraud_cases,
    SUM(isFraud) / COUNT(*) as Fraud_rate,
    CAST((MAX(TransactionDT) - MIN(TransactionDT)) / 86400 AS INT) as Time_span_days,
    COUNT(id_01) as Identity_coverage
FROM t
LEFT JOIN i
ON t.TransactionID = i.TransactionID
"""
df_summary = ctx.execute(query).collect()
display(Markdown(df_summary.to_pandas().to_markdown(index=False)))

|   Transactions |   Fraud_cases |   Fraud_rate |   Time_span_days |   Identity_coverage |
|---------------:|--------------:|-------------:|-----------------:|--------------------:|
|         590540 |         20663 |      0.03499 |              181 |              144233 |

In [3]:
query_drift = """
SELECT 
    CAST((TransactionDT - 86400) / 2592000 AS INTEGER) as Bucket,
    COUNT(*) as Transactions,
    SUM(isFraud) / COUNT(*) as Fraud_rate,
    AVG(TransactionAmt) as Mean_amount
FROM t
GROUP BY Bucket
ORDER BY Bucket
"""
df_drift = ctx.execute(query_drift).collect()

# Format df_drift to match markdown table
df_drift_md = df_drift.with_columns([
    pl.col("Bucket").map_elements(lambda x: f"{x} (partial)" if x == 6 else str(x), return_dtype=pl.String),
    pl.col("Fraud_rate").map_elements(lambda x: f"**{x*100:.2f}%**" if x < 0.03 or x > 0.043 else f"{x*100:.2f}%", return_dtype=pl.String),
    pl.col("Mean_amount").map_elements(lambda x: f"{x:.2f}", return_dtype=pl.String)
])
display(Markdown(df_drift_md.to_pandas().to_markdown(index=False)))


| Bucket      |   Transactions | Fraud_rate   |   Mean_amount |
|:------------|---------------:|:-------------|--------------:|
| 0           |         134339 | **2.53%**    |        128.28 |
| 1           |          89399 | 4.00%        |        133.53 |
| 2           |          92189 | 4.04%        |        140.03 |
| 3           |          98615 | 3.95%        |        139.53 |
| 4           |          83571 | 3.41%        |        133.7  |
| 5           |          86934 | 3.42%        |        136.1  |
| 6 (partial) |           5493 | **4.39%**    |        162.89 |

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_daily = t.with_columns(
    ((pl.col("TransactionDT") - 86400) // 86400).cast(pl.Int32).alias("Day")
).group_by("Day").agg([
    pl.len().alias("Transactions"),
    (pl.col("isFraud").sum() / pl.len()).alias("Fraud_rate"),
    pl.col("TransactionAmt").median().alias("Median_amount"),
    pl.col("TransactionAmt").quantile(0.95).alias("P95_amount")
]).sort("Day").collect().to_pandas()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, specs=[[{"secondary_y": True}], [{"secondary_y": False}]])

fig.add_trace(go.Bar(x=df_daily["Day"], y=df_daily["Transactions"], name="Daily Volume", opacity=0.3, marker_color="gray"), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=df_daily["Day"], y=df_daily["Fraud_rate"], name="Fraud Rate", mode="lines", line={"color": "red"}), row=1, col=1, secondary_y=True)

fig.add_trace(go.Scatter(x=df_daily["Day"], y=df_daily["Median_amount"], name="Median Amount", mode="lines"), row=2, col=1)
fig.add_trace(go.Scatter(x=df_daily["Day"], y=df_daily["P95_amount"], name="95th Percentile Amount", mode="lines"), row=2, col=1)

fig.update_layout(
    title="Daily Transactions, Fraud Rate & Amount Percentiles",
    hovermode="x unified",
    template="plotly_white",
    height=600,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "right", "x": 1}
)
fig.update_yaxes(title_text="Transactions Count", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Fraud Rate", tickformat=".1%", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="Amount (USD)", row=2, col=1)
fig.update_xaxes(title_text="Day (from start of dataset)", row=2, col=1)
fig.show()


### Transaction Amount vs Fraud

In [5]:
# Calculate Amount vs Fraud metrics using Polars lazyframe
amt_stats = t.select([
    pl.col("TransactionAmt").median().alias("Median"),
    pl.col("TransactionAmt").quantile(0.95).alias("95th percentile"),
    pl.col("TransactionAmt").max().alias("Maximum"),
]).collect().row(0)

mean_fraud = t.filter(pl.col("isFraud") == 1).select(pl.col("TransactionAmt").mean()).collect().item()
mean_legit = t.filter(pl.col("isFraud") == 0).select(pl.col("TransactionAmt").mean()).collect().item()

amt_data = [
    {"Metric": "Median", "Value": f"{amt_stats[0]:.2f}"},
    {"Metric": "95th percentile", "Value": f"{amt_stats[1]:.2f}"},
    {"Metric": "Maximum", "Value": f"{amt_stats[2]:,.2f}"},
    {"Metric": "Mean, fraud", "Value": f"**{mean_fraud:.2f}**"},
    {"Metric": "Mean, legitimate", "Value": f"**{mean_legit:.2f}**"},
    {"Metric": "Gap between fraud and legitimate means", "Value": f"**+{mean_fraud - mean_legit:.2f} (+{(mean_fraud - mean_legit)/mean_legit * 100:.1f}%)**"}
]
display(Markdown(pl.DataFrame(amt_data).to_pandas().to_markdown(index=False)))


| Metric                                 | Value               |
|:---------------------------------------|:--------------------|
| Median                                 | 68.77               |
| 95th percentile                        | 445.00              |
| Maximum                                | 31,937.39           |
| Mean, fraud                            | **149.24**          |
| Mean, legitimate                       | **134.51**          |
| Gap between fraud and legitimate means | **+14.73 (+11.0%)** |

The gap between fraudulent and legitimate means is ~11%.
Amount alone separates these classes very weakly.

Because fraudulent amounts are not concentrated in a long tail, a cost function driven by amount will not simplify into a rule to block large transactions.

---
